<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day5/Session4/session4_capstone.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — Providers, Cost, Safety & Capstone

**Day 5 · 15:30 – 16:30**

Compare providers, estimate a real bill, audit your own security, and finish the week.

## Cell 1 — Setup

In [ ]:
!pip install -q -U google-genai

import os, json, time
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client()
MODEL = "gemini-3.5-flash"

print("Client ready.")

## Cell 2 — Estimate what a real deployment costs

You will be asked this question by a manager one day. Here is how to answer it.

In [ ]:
def estimate_monthly_cost(requests_per_day,
                          input_tokens_per_request,
                          output_tokens_per_request,
                          price_per_1m_input=1.50,
                          price_per_1m_output=7.50):
    """Estimate a monthly API bill.

    Prices change constantly - always check the current pricing page.
    The METHOD is what matters: count requests, count tokens, multiply.
    """
    reqs = requests_per_day * 30
    in_tok = reqs * input_tokens_per_request
    out_tok = reqs * output_tokens_per_request

    in_cost = in_tok / 1_000_000 * price_per_1m_input
    out_cost = out_tok / 1_000_000 * price_per_1m_output

    print(f"  requests / month : {reqs:>12,}")
    print(f"  input tokens     : {in_tok:>12,}   ${in_cost:>8.2f}")
    print(f"  output tokens    : {out_tok:>12,}   ${out_cost:>8.2f}")
    print(f"  {'-'*44}")
    print(f"  ESTIMATED TOTAL                    ${in_cost + out_cost:>8.2f} / month")
    return in_cost + out_cost


print("SCENARIO A - small department help desk")
estimate_monthly_cost(requests_per_day=200,
                      input_tokens_per_request=500,
                      output_tokens_per_request=300)

print()
print("SCENARIO B - university-wide student assistant")
estimate_monthly_cost(requests_per_day=5000,
                      input_tokens_per_request=800,
                      output_tokens_per_request=400)

### The lesson

Scenario A is affordable on a modest budget. Scenario B is a real financial decision
that needs approval.

**Notice what drives the cost:** output tokens are typically far more expensive than
input tokens. Asking for shorter answers is one of the cheapest optimisations available
— and a system instruction does it for free.

## Cell 3 — When the free tier stops being enough

In [ ]:
def days_until_quota_exhausted(requests_per_day, daily_free_limit):
    if requests_per_day <= daily_free_limit:
        return "Fits inside the free tier indefinitely"
    hours = daily_free_limit / requests_per_day * 24
    return f"Free quota exhausted after about {hours:.1f} hours each day"


for rpd, limit in [(50, 250), (200, 250), (1000, 250), (5000, 250)]:
    print(f"{rpd:>5} requests/day vs {limit} free : {days_until_quota_exhausted(rpd, limit)}")

print()
print("Free-tier limits change often.")
print("Check: https://ai.google.dev/gemini-api/docs/rate-limits")

> ### Three things to check before choosing free tier for real work
> 1. **Daily cap** — will your usage fit?
> 2. **Data policy** — free-tier prompts may be used to improve the model. Read the terms.
> 3. **Availability** — free tiers carry no uptime guarantee.
>
> For anything sensitive, that second point alone rules out the free tier. That is
> exactly what Day 4 was for: a fine-tuned small model on your own hardware sends
> nothing anywhere.

## Cell 4 — Security audit of your own notebook

Every point below is a real incident that happens somewhere every week.

In [ ]:
import re

def audit_notebook_text(text):
    """Check a block of code/text for common credential mistakes."""
    problems = []

    if re.search(r"AIzaSy[A-Za-z0-9_\-]{10,}", text):
        problems.append("A Google API key appears in plain text")
    if re.search(r"sk-[A-Za-z0-9]{20,}", text):
        problems.append("An OpenAI-style key appears in plain text")
    if re.search(r'(api_key|API_KEY)\s*=\s*["\'][^"\']{15,}["\']', text):
        problems.append("A key is hard-coded in an assignment")

    if problems:
        print("PROBLEMS FOUND:")
        for p in problems:
            print("  X", p)
    else:
        print("OK - no hard-coded credentials found in this text")
    return problems


print("--- example of BAD code ---")
audit_notebook_text('client = genai.Client(api_key="AIzaSyD-EXAMPLE-not-a-real-key-12345")')

print()
print("--- example of GOOD code ---")
audit_notebook_text("client = genai.Client()   # reads GEMINI_API_KEY from environment")

### Your manual checklist

Run through this before sharing anything:

- [ ] Search the notebook for `AIzaSy` — if it appears, the key is exposed
- [ ] Every key comes from `userdata.get()` or `os.environ`
- [ ] **Clear all outputs** before sharing — keys can sit in old output cells
- [ ] No real personal data was pasted into a prompt while learning
- [ ] If a key ever leaked: delete it in AI Studio and create a new one, immediately

> Deleting a leaked key takes 30 seconds. Explaining an unexpected bill takes longer.

## Cell 5 — Switching providers is easier than you think

Most providers copy OpenAI's request format, so moving is often a two-line change.
This makes your skills portable and protects you from lock-in.

In [ ]:
PROVIDERS = {
    "Google Gemini": {
        "free_tier": "Yes, no credit card",
        "get_key":   "aistudio.google.com/apikey",
        "strength":  "Huge context window, built-in search grounding",
    },
    "Groq": {
        "free_tier": "Yes",
        "get_key":   "console.groq.com",
        "strength":  "Very fast responses, runs open models",
    },
    "OpenRouter": {
        "free_tier": "Some free models",
        "get_key":   "openrouter.ai",
        "strength":  "One key, hundreds of models, easy comparison",
    },
    "Hugging Face": {
        "free_tier": "Yes",
        "get_key":   "huggingface.co/settings/tokens",
        "strength":  "Open models - including the one you fine-tuned on Day 4",
    },
    "Ollama (local)": {
        "free_tier": "Free forever - it is your computer",
        "get_key":   "no key needed",
        "strength":  "Works offline, data never leaves the machine",
    },
}

for name, info in PROVIDERS.items():
    print(f"{name}")
    for k, v in info.items():
        print(f"    {k:<10}: {v}")
    print()

## Cell 6 — THE CAPSTONE

Build one assistant that would genuinely help someone in your department.

**Requirements:**
1. A clear system instruction defining its role
2. Answers grounded in a document *or* live search
3. Refuses politely when it does not know
4. Error handling that survives a rate limit
5. Works with questions in English **and** Amharic

In [ ]:
# ============================================================
# YOUR CAPSTONE - edit everything below
# ============================================================

ASSISTANT_NAME = "EDU Study Helper"

KNOWLEDGE = """
Replace this with your own content:
  - a course outline
  - departmental procedures
  - a reading list
  - equipment instructions
"""

SYSTEM_INSTRUCTION = (
    f"You are {ASSISTANT_NAME}, an assistant for the Ethiopian Defense University. "
    "Answer using ONLY the KNOWLEDGE section provided. "
    "If the answer is not there, say so plainly and suggest who to ask. "
    "Reply in Amharic if the question is in Amharic, otherwise in English. "
    "Keep answers under 100 words. Never invent facts."
)


def capstone_assistant(question, tries=3):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        max_output_tokens=400,
    )
    prompt = f"KNOWLEDGE:\n{KNOWLEDGE}\n\nQUESTION: {question}"

    for attempt in range(tries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt, config=config)
            return r.text.strip()
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < tries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"Sorry, something went wrong: {str(e)[:120]}"
    return "Sorry, the service is busy. Please try again shortly."


# --- test it, including a question it should refuse ---
tests = [
    "What is this course about?",
    "ይህ ኮርስ ስለ ምንድን ነው?",
    "What is the price of gold today?",     # should be refused
]

for q in tests:
    print("Q:", q)
    print("A:", capstone_assistant(q))
    print("-" * 60)
    time.sleep(1)

## Cell 7 — Save your capstone and push it to GitHub

In [ ]:
capstone_code = f'''"""
{ASSISTANT_NAME}
Built during Day 5 of the EDU AI Lab training.

Setup:
    pip install google-genai
    export GEMINI_API_KEY="your-key-here"

Run:
    python capstone_assistant.py
"""

import os, time
from google import genai
from google.genai import types

client = genai.Client()
MODEL = "{MODEL}"

KNOWLEDGE = """{KNOWLEDGE}"""

SYSTEM_INSTRUCTION = "{SYSTEM_INSTRUCTION}"


def assistant(question, tries=3):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        max_output_tokens=400)
    prompt = f"KNOWLEDGE:\\n{{KNOWLEDGE}}\\n\\nQUESTION: {{question}}"

    for attempt in range(tries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt, config=config)
            return r.text.strip()
        except Exception as e:
            if "429" in str(e) and attempt < tries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"Error: {{str(e)[:120]}}"
    return "Service busy, please retry."


if __name__ == "__main__":
    print("{ASSISTANT_NAME} - type 'quit' to exit")
    while True:
        q = input("\\nYou: ")
        if q.lower() in ("quit", "exit"):
            break
        print("Assistant:", assistant(q))
'''

with open("capstone_assistant.py", "w") as f:
    f.write(capstone_code)

print("Saved capstone_assistant.py")
print()
print("NOTE: the file reads the key from the environment - no key inside it.")
print()
print("To publish:")
print("  1. Download it from the file panel on the left")
print("  2. Add it to your EDU-AI-Training repository under Day5/")
print("  3. Commit and push")

## Cell 8 — Final check before you leave

In [ ]:
print("=" * 60)
print("  DAY 5 FINAL CHECK")
print("=" * 60)

checks = [
    "API key created and stored in Colab Secrets (not in code)",
    "Made API calls with a system instruction",
    "Streamed a response",
    "Built a multi-turn chat",
    "Parsed structured JSON output",
    "Handled a 429 rate-limit error without crashing",
    "Built an assistant that answers from a document",
    "It refuses questions it cannot answer",
    "Grounded an answer in live web search with sources",
    "Audited the notebook for leaked keys",
    "Saved the capstone and pushed it to GitHub",
]

for c in checks:
    print(f"  [ ] {c}")

print()
print("=" * 60)
print("  Clear all outputs before sharing this notebook.")
print("  Edit > Clear all outputs")
print("=" * 60)

---

# The whole week

```
DAY 1   The AI ecosystem, GitHub, Colab, Hugging Face, Copilot
DAY 2   Local Python and cloud AI platforms
DAY 3   Microsoft Azure: guest access, storage, ML workspace, AI services
DAY 4   One neuron -> 494M parameters -> LoRA fine-tuning for Amharic
DAY 5   APIs: rent a large model, control it, ground it, ship it
```

## Three ideas that ran through all five days

1. **Know what things cost.** GPU memory, cloud bills, API tokens. Every technical
   decision is also a budget decision.
2. **Prove what you claim.** Baseline before fine-tuning. Cite your sources. Say what
   did *not* improve.
3. **Choose the smallest tool that solves the problem.** Prompt before RAG. RAG before
   fine-tuning. API before buying hardware — unless you need offline or privacy, and
   then the reverse.

## What you can now do

Build, fine-tune, deploy and integrate AI systems — using tools that were free or
nearly free the entire week.

**Now go and build something your institution actually needs.**